# stg_weather_v2 — Silver Cleaning for Open-Meteo Weather Data
## Medallion Architecture: Bronze → Silver

**MSBA 305 — Google Colab Version**

**What this notebook does:**
Reads the Bronze Delta table `bronze_weather_nyc`, applies all 9 cleaning
and validation steps and writes the result
to `silver_stg_weather`.

Suspicious but possibly legitimate rows (extreme weather events) are routed to
`quarantine_weather` for human review — never deleted.

### Changelog from v1
| Change | Reason |
|--------|--------|
| Read from Delta path on Drive | Colab version — no managed tables |
| Column name `timestamp` (not `time`) | Matches ingestion notebook schema |
| Write via `.save()` to Drive path | Colab version — replaces `saveAsTable()` |
| Snowfall conversion ÷7 (not ÷10) | v1 Bronze used ÷10 incorrectly; Silver recalculates |
| Added "Cloudy" category (codes 4–49 excl fog) | Was missing in original v1 |
| Added snow shower codes 85, 86 to Snow category | WMO codes for snow showers |
| Skip redundant timestamp parsing | Bronze already casts to TimestampType |

### Steps
| Step | What it does | Hard fail or quarantine? |
|------|-------------|------------------------|
| 1–2 | Flatten JSON + add borough | Already done in Bronze ingestion |
| 3 | Verify timestamps parsed, check hourly continuity | Hard fail |
| 4 | Single-column NULL + range checks | Hard fail |
| 5 | Derive snowfall_mm (÷7) | Transformation |
| 6 | Cross-field validation | Hard fail (with tolerance) |
| 7 | Quarantine outliers | Soft — route to quarantine table |
| 8 | Create weather_category | Transformation |
| 9 | Write to Silver Delta table | Final write |


In [107]:
import datetime, uuid
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from notebookutils import mssparkutils
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Runtime identifiers
today = datetime.date.today()
now = datetime.datetime.utcnow()
ingest_run_id = str(uuid.uuid4())

# Table names only (Fabric manages the rest)
BRONZE_TABLE = "bronze_weather_nyc"
SILVER_TABLE = "silver_stg_weather"
QUARANTINE_TABLE = "silver_quarantine_weather"

StatementMeta(, 02149e88-cb61-466f-89b6-c94d46ee6b43, 109, Finished, Available, Finished, False)

In [108]:
# Valid WMO weather codes used by Open-Meteo
VALID_WMO_CODES = {
    0, 1, 2, 3,          # Clear / Mainly clear / Partly cloudy / Overcast
    45, 48,              # Fog / Depositing rime fog
    51, 53, 55,          # Drizzle: light / moderate / dense
    56, 57,              # Freezing drizzle: light / dense
    61, 63, 65,          # Rain: slight / moderate / heavy
    66, 67,              # Freezing rain: light / heavy
    71, 73, 75,          # Snow fall: slight / moderate / heavy
    77,                  # Snow grains
    80, 81, 82,          # Rain showers: slight / moderate / violent
    85, 86,              # Snow showers: slight / heavy
    95,                  # Thunderstorm: slight or moderate
    96, 99,              # Thunderstorm with slight / heavy hail
}

print("Configuration loaded")
print(f"   Bronze source:    {BRONZE_TABLE}")
print(f"   Silver target:    {SILVER_TABLE}")
print(f"   Quarantine table: {QUARANTINE_TABLE}")

StatementMeta(, 02149e88-cb61-466f-89b6-c94d46ee6b43, 110, Finished, Available, Finished, False)

Configuration loaded
   Bronze source:    bronze_weather_nyc
   Silver target:    silver_stg_weather
   Quarantine table: silver_quarantine_weather


In [109]:
# ═══════════════════════════════════════════════════════════════════════════════
# ONE-TIME PURGE — run once to start fresh, then delete or skip this cell
# ═══════════════════════════════════════════════════════════════════════════════
'''
for tbl in [SILVER_TABLE, QUARANTINE_TABLE]:
    try:
        spark.sql(f"TRUNCATE TABLE {tbl}")
        print(f" Truncated: {tbl}")
    except Exception as e:
        print(f"  {tbl} doesn't exist yet, skipping: {e}")

print("\nPurge complete — re-run pipeline from Cell 4.")
'''

StatementMeta(, 02149e88-cb61-466f-89b6-c94d46ee6b43, 111, Finished, Available, Finished, False)

'\nfor tbl in [SILVER_TABLE, QUARANTINE_TABLE]:\n    try:\n        spark.sql(f"TRUNCATE TABLE {tbl}")\n        print(f"✅ Truncated: {tbl}")\n    except Exception as e:\n        print(f"⚠️  {tbl} doesn\'t exist yet, skipping: {e}")\n\nprint("\nPurge complete — re-run pipeline from Cell 4.")\n'

In [110]:

"""
# ── Delete last 50 rows from Silver (for incremental write testing) ───────────

from delta.tables import DeltaTable

# Find the 50 most recent timestamps
cutoff = (
    spark.table(SILVER_TABLE)
    .orderBy(F.col("timestamp").desc())
    .limit(50)
    .select("borough", "timestamp")
)

# Show what will be deleted before committing
print("Rows to be deleted:")
cutoff.show(10, truncate=False)

# Delete via Delta merge
dt = DeltaTable.forName(spark, SILVER_TABLE)
dt.delete(
    F.struct("borough", "timestamp").isin(
        [F.struct(F.lit(r["borough"]), F.lit(r["timestamp"])) for r in cutoff.collect()]
    )
)

after_count = spark.table(SILVER_TABLE).count()
print(f" Deleted 50 rows — Silver now has {after_count:,} rows")
print(f"   Re-run the pipeline and confirm it writes exactly 50 rows back.")

"""

StatementMeta(, 02149e88-cb61-466f-89b6-c94d46ee6b43, 112, Finished, Available, Finished, False)

'\n# ── Delete last 50 rows from Silver (for incremental write testing) ───────────\n\nfrom delta.tables import DeltaTable\n\n# Find the 50 most recent timestamps\ncutoff = (\n    spark.table(SILVER_TABLE)\n    .orderBy(F.col("timestamp").desc())\n    .limit(50)\n    .select("borough", "timestamp")\n)\n\n# Show what will be deleted before committing\nprint("Rows to be deleted:")\ncutoff.show(10, truncate=False)\n\n# Delete via Delta merge\ndt = DeltaTable.forName(spark, SILVER_TABLE)\ndt.delete(\n    F.struct("borough", "timestamp").isin(\n        [F.struct(F.lit(r["borough"]), F.lit(r["timestamp"])) for r in cutoff.collect()]\n    )\n)\n\nafter_count = spark.table(SILVER_TABLE).count()\nprint(f"✅ Deleted 50 rows — Silver now has {after_count:,} rows")\nprint(f"   Re-run the pipeline and confirm it writes exactly 50 rows back.")\n\n'

## Steps 1–2: Flatten JSON + Add Borough Column
> **Already done in the Bronze ingestion notebook.**
>
> This notebook picks up from the Bronze Delta table — each row is already one hour × one borough.

In [111]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEPS 1–2: Read Bronze Delta table from Google Drive
# ── COLAB CHANGE: spark.read.format("delta").load() instead of spark.table() ──
# ═══════════════════════════════════════════════════════════════════════════════

df = spark.table(BRONZE_TABLE)

row_count = df.count()
print(f"Bronze weather loaded: {row_count:,} rows")
print(f"   Columns ({len(df.columns)}): {df.columns}")
print(f"   Boroughs: {[r[0] for r in df.select('borough').distinct().collect()]}")
df.printSchema()

assert row_count > 0, "PIPELINE ERROR: Bronze weather table is empty!"

print(f"\nSteps 1–2 complete: Bronze table loaded with {row_count:,} rows")

StatementMeta(, 02149e88-cb61-466f-89b6-c94d46ee6b43, 113, Finished, Available, Finished, False)

Bronze weather loaded: 13,320 rows
   Columns (26): ['timestamp', 'temperature_2m', 'apparent_temperature', 'relative_humidity_2m', 'dew_point_2m', 'precipitation', 'rain', 'snowfall', 'snow_depth', 'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m', 'cloud_cover', 'pressure_msl', 'weather_code', 'latitude', 'longitude', 'timezone', 'timezone_abbreviation', 'elevation', 'borough', 'source', 'ingest_run_id', 'ingested_at', 'ingest_date', 'year']
   Boroughs: ['bronx', 'statenisland', 'manhattan', 'queens', 'brooklyn']
root
 |-- timestamp: timestamp (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- apparent_temperature: double (nullable = true)
 |-- relative_humidity_2m: double (nullable = true)
 |-- dew_point_2m: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- snowfall: double (nullable = true)
 |-- snow_depth: double (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- wind_direction

## Step 3: Deduplicate, Verify Timestamps, and Check Hourly Continuity
> Bronze ingestion already parses timestamps. Here we verify, deduplicate, and check continuity.

In [112]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: Deduplicate + verify timestamps + check hourly continuity
# ═══════════════════════════════════════════════════════════════════════════════

# 3a: Check type and NULLs
ts_dtype = dict(df.dtypes).get("timestamp")
print(f"   timestamp column type: {ts_dtype}")
if ts_dtype != "timestamp":
    print("   timestamp is not TimestampType — attempting to cast...")
    df = df.withColumn("timestamp", F.to_timestamp("timestamp"))

null_times = df.filter(F.col("timestamp").isNull()).count()
if null_times > 0:
    raise Exception(
        f"PIPELINE ERROR: {null_times} NULL timestamps found. "
        "Check the Bronze ingestion — some time strings could not be parsed."
    )
print(f"Step 3a: All timestamps are valid (0 NULLs)")

# 3b: Deduplicate — keep the latest-ingested row per (borough, timestamp)
pre_dedup = df.count()

if "ingested_at" in df.columns:
    w_dedup = Window.partitionBy("borough", "timestamp").orderBy(F.desc("ingested_at"))
    df = (df.withColumn("_row_num", F.row_number().over(w_dedup))
       .filter(F.col("_row_num") == 1)
       .drop("_row_num"))
else:
    df = df.dropDuplicates(["borough", "timestamp"])

post_dedup = df.count()
dupes_removed = pre_dedup - post_dedup
print(f"Step 3b: Deduplication complete")
print(f"   Before: {pre_dedup:,} rows")
print(f"   After:  {post_dedup:,} rows")
print(f"   Duplicates removed: {dupes_removed:,}")

# 3c: Verify hourly continuity within each borough
w = Window.partitionBy("borough").orderBy("timestamp")

df_with_gaps = (
    df.withColumn("_prev_time", F.lag("timestamp").over(w))
      .withColumn("_gap_hours",
                  (F.unix_timestamp("timestamp") - F.unix_timestamp("_prev_time")) / 3600)
)

bad_gaps = df_with_gaps.filter(
    F.col("_prev_time").isNotNull() &
    (~F.col("_gap_hours").isin([1.0, 2.0, 0.0]))
)
bad_gap_count = bad_gaps.count()

if bad_gap_count > 0:
    print(f"{bad_gap_count} unexpected timestamp gaps found:")
    bad_gaps.select("borough", "timestamp", "_prev_time", "_gap_hours").show(20, truncate=False)
    raise Exception(
        f"PIPELINE ERROR: {bad_gap_count} unexpected timestamp gaps found. "
        "Expected 1h gaps (normal), 2h (spring DST), or 0h (fall DST)."
    )

# Report DST transitions if any
dst_spring = df_with_gaps.filter(F.col("_gap_hours") == 2.0).count()
dst_fall = df_with_gaps.filter(F.col("_gap_hours") == 0.0).count()
if dst_spring > 0 or dst_fall > 0:
    print(f"   DST transitions detected: {dst_spring} spring-forward, {dst_fall} fall-back")

df = df_with_gaps.drop("_prev_time", "_gap_hours")

time_range = df.agg(F.min("timestamp").alias("earliest"), F.max("timestamp").alias("latest")).collect()[0]
print(f"Step 3c: Hourly continuity verified")
print(f"   Time range: {time_range['earliest']} -> {time_range['latest']}")

StatementMeta(, 02149e88-cb61-466f-89b6-c94d46ee6b43, 114, Finished, Available, Finished, False)

   timestamp column type: timestamp
Step 3a: All timestamps are valid (0 NULLs)
Step 3b: Deduplication complete
   Before: 13,320 rows
   After:  13,320 rows
   Duplicates removed: 0
Step 3c: Hourly continuity verified
   Time range: 2026-01-01 00:00:00 -> 2026-04-21 23:00:00


## Step 4: Single-Column Constraints (NULL Checks + Range Checks)

In [113]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4a: NULL checks on all weather fields
# ═══════════════════════════════════════════════════════════════════════════════

NOT_NULL_FIELDS = [
    "temperature_2m", "apparent_temperature", "dew_point_2m",
    "relative_humidity_2m", "rain", "snowfall", "snow_depth",
    "wind_speed_10m", "wind_direction_10m", "wind_gusts_10m",
    "cloud_cover", "precipitation", "pressure_msl", "weather_code",
]

print("Running NULL checks on all weather fields...")
for field in NOT_NULL_FIELDS:
    if field not in df.columns:
        raise Exception(
            f"SCHEMA ERROR: Column '{field}' not found in Bronze table. "
            f"Available columns: {df.columns}. "
            "Check the ingestion notebook — all 14 hourly fields must be included."
        )
    null_count = df.filter(F.col(field).isNull()).count()
    if null_count > 0:
        raise Exception(
            f"PIPELINE ERROR: {null_count} NULLs in '{field}'. "
            "ERA5 has no incomplete fields — this is a pipeline error."
        )
    print(f"   {field}: 0 NULLs")

print(f"\nStep 4a complete: all {len(NOT_NULL_FIELDS)} fields passed NULL checks")

StatementMeta(, 02149e88-cb61-466f-89b6-c94d46ee6b43, 115, Finished, Available, Finished, False)

Running NULL checks on all weather fields...
   temperature_2m: 0 NULLs
   apparent_temperature: 0 NULLs
   dew_point_2m: 0 NULLs
   relative_humidity_2m: 0 NULLs
   rain: 0 NULLs
   snowfall: 0 NULLs
   snow_depth: 0 NULLs
   wind_speed_10m: 0 NULLs
   wind_direction_10m: 0 NULLs
   wind_gusts_10m: 0 NULLs
   cloud_cover: 0 NULLs
   precipitation: 0 NULLs
   pressure_msl: 0 NULLs
   weather_code: 0 NULLs

Step 4a complete: all 14 fields passed NULL checks


In [114]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4b: Single-column range checks
# ═══════════════════════════════════════════════════════════════════════════════

HARD_RANGE_CHECKS = {
    "relative_humidity_2m":  (0,    100,  "Humidity is a percentage (0-100)"),
    "rain":                  (0,    None, "Rainfall cannot be negative"),
    "snowfall":              (0,    None, "Snowfall cannot be negative"),
    "snow_depth":            (0,    None, "Snow depth cannot be negative"),
    "wind_speed_10m":        (0,    None, "Wind speed cannot be negative"),
    "wind_direction_10m":    (0,    360,  "Wind direction is 0-360 degrees"),
    "cloud_cover":           (0,    100,  "Cloud cover is a percentage (0-100)"),
    "precipitation":         (0,    None, "Precipitation cannot be negative"),
    "pressure_msl":          (950,  1060, "Pressure outside 950-1060 hPa is a data error"),
}

print("Running range checks...")
for field, (low, high, reason) in HARD_RANGE_CHECKS.items():
    violation = F.lit(False)
    if low is not None:
        violation = violation | (F.col(field) < low)
    if high is not None:
        violation = violation | (F.col(field) > high)

    bad_rows = df.filter(violation).count()
    if bad_rows > 0:
        print(f"\n{field} -- {bad_rows} violations found:")
        df.filter(violation).select("timestamp", "borough", field).show(5, truncate=False)
        raise Exception(
            f"PIPELINE ERROR: {bad_rows} rows in '{field}' outside [{low}, {high}]. "
            f"{reason}."
        )
    print(f"   {field}: all values in [{low}, {high}]")

print(f"\nStep 4b complete: all range checks passed")

StatementMeta(, 02149e88-cb61-466f-89b6-c94d46ee6b43, 116, Finished, Available, Finished, False)

Running range checks...
   relative_humidity_2m: all values in [0, 100]
   rain: all values in [0, None]
   snowfall: all values in [0, None]
   snow_depth: all values in [0, None]
   wind_speed_10m: all values in [0, None]
   wind_direction_10m: all values in [0, 360]
   cloud_cover: all values in [0, 100]
   precipitation: all values in [0, None]
   pressure_msl: all values in [950, 1060]

Step 4b complete: all range checks passed


## Step 5: Derive `snowfall_mm`

In [115]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: Derive snowfall_mm from snowfall (cm)
# snowfall_mm = snowfall_cm / 7  (standard snow-to-water ratio)
# ═══════════════════════════════════════════════════════════════════════════════

df = df.withColumn("snowfall_mm", F.round(F.col("snowfall") / 7, 4))

snow_sample = df.filter(F.col("snowfall") > 0).select(
    "timestamp", "borough", "snowfall", "snowfall_mm"
)
snow_count = snow_sample.count()
print(f"Step 5: snowfall_mm derived (div 7) -- {snow_count} rows have non-zero snowfall")
if snow_count > 0:
    print("   Sample conversions (cm -> mm water equivalent):")
    snow_sample.show(5, truncate=False)
else:
    print("   (No snowfall recorded in this date range — all values are 0)")

StatementMeta(, 02149e88-cb61-466f-89b6-c94d46ee6b43, 117, Finished, Available, Finished, False)

Step 5: snowfall_mm derived (div 7) -- 615 rows have non-zero snowfall
   Sample conversions (cm -> mm water equivalent):
+-------------------+-------+--------+-----------+
|timestamp          |borough|snowfall|snowfall_mm|
+-------------------+-------+--------+-----------+
|2026-01-01 01:00:00|bronx  |0.07    |0.01       |
|2026-01-01 02:00:00|bronx  |0.07    |0.01       |
|2026-01-01 07:00:00|bronx  |0.14    |0.02       |
|2026-01-01 08:00:00|bronx  |0.56    |0.08       |
|2026-01-01 09:00:00|bronx  |0.07    |0.01       |
+-------------------+-------+--------+-----------+
only showing top 5 rows



## Step 6: Cross-Field Validation (Business Rule Checks)

In [116]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: Cross-field validation
# ═══════════════════════════════════════════════════════════════════════════════

# 6a: Dew point must be <= temperature (0.5C tolerance)
dew_violations = df.filter(F.col("dew_point_2m") > F.col("temperature_2m") + 0.5).count()
if dew_violations > 0:
    print("Dew point > temperature violations (beyond 0.5C tolerance):")
    df.filter(F.col("dew_point_2m") > F.col("temperature_2m") + 0.5) \
      .select("timestamp", "borough", "dew_point_2m", "temperature_2m").show(5, truncate=False)
    raise Exception(
        f"PIPELINE ERROR: {dew_violations} rows where dew_point_2m > temperature_2m + 0.5C. "
        "This violates thermodynamic law beyond rounding tolerance."
    )

minor_dew = df.filter(
    (F.col("dew_point_2m") > F.col("temperature_2m")) &
    (F.col("dew_point_2m") <= F.col("temperature_2m") + 0.5)
).count()
if minor_dew > 0:
    print(f"   {minor_dew} rows where dew_point exceeds temperature by <=0.5C (ERA5 rounding — accepted)")
print(f"Step 6a: dew_point <= temperature -- passed")


# 6b: Wind gusts must be >= wind speed
# If gust < speed it is a sensor/averaging artifact — correct it in-place (gust = speed)
gust_violations_df = df.filter(F.col("wind_gusts_10m") < F.col("wind_speed_10m"))
gust_violations = gust_violations_df.count()

if gust_violations > 0:
    print(f"  Step 6b: {gust_violations} rows where wind_gusts_10m < wind_speed_10m")
    print("   Correcting: setting wind_gusts_10m = wind_speed_10m for affected rows")
    gust_violations_df.select(
        "timestamp", "borough", "wind_gusts_10m", "wind_speed_10m"
    ).show(10, truncate=False)

    df = df.withColumn(
        "wind_gusts_10m",
        F.when(
            F.col("wind_gusts_10m") < F.col("wind_speed_10m"),
            F.col("wind_speed_10m")
        ).otherwise(F.col("wind_gusts_10m"))
    )
    print(f"    Correction applied — pipeline continues")
else:
    print("Step 6b: wind_gusts_10m >= wind_speed_10m -- passed")


# 6c: precipitation consistency check
# NOTE: Open-Meteo precipitation = rain + showers + snowfall_water_equivalent
# 'showers' was not ingested as a separate column, so rain + snowfall_mm will
# always be ≤ precipitation. This check logs the gap for awareness but never
# breaks the pipeline — the deviation is expected, not a data error.

df = df.withColumn(
    "_precip_diff",
    F.col("precipitation") - (F.col("rain") + F.col("snowfall_mm"))
)

negative_precip = df.filter(F.col("_precip_diff") < -0.5).count()
if negative_precip > 0:
    # This IS a real error — rain + snowfall_mm cannot exceed total precipitation
    raise Exception(
        f"PIPELINE ERROR: {negative_precip} rows where rain + snowfall_mm "
        f"EXCEEDS precipitation by >0.5mm. This violates conservation — check ingestion."
    )

over_threshold = df.filter(F.col("_precip_diff") > 0.5).count()
total_rows = df.count()
if over_threshold > 0:
    print(f"Step 6c:   {over_threshold:,} rows ({over_threshold/total_rows*100:.1f}%) "
          f"where precipitation > rain + snowfall_mm by >0.5mm")
    print(f"   Expected — gap is accounted for by 'showers' (not ingested as separate column)")
    print(f"   Pipeline continues.")
else:
    print(f"Step 6c: precipitation >= rain + snowfall_mm — passed")

df = df.drop("_precip_diff")

StatementMeta(, 02149e88-cb61-466f-89b6-c94d46ee6b43, 118, Finished, Available, Finished, False)

   4 rows where dew_point exceeds temperature by <=0.5C (ERA5 rounding — accepted)
Step 6a: dew_point <= temperature -- passed
⚠️  Step 6b: 4 rows where wind_gusts_10m < wind_speed_10m
   Correcting: setting wind_gusts_10m = wind_speed_10m for affected rows
+-------------------+------------+--------------+--------------+
|timestamp          |borough     |wind_gusts_10m|wind_speed_10m|
+-------------------+------------+--------------+--------------+
|2026-02-18 20:00:00|bronx       |5.8           |6.0           |
|2026-01-07 08:00:00|queens      |1.8           |2.6           |
|2026-01-14 20:00:00|statenisland|9.0           |9.2           |
|2026-01-18 08:00:00|statenisland|2.5           |3.2           |
+-------------------+------------+--------------+--------------+

   ✅ Correction applied — pipeline continues
Step 6c: ⚠️  243 rows (1.8%) where precipitation > rain + snowfall_mm by >0.5mm
   Expected — gap is accounted for by 'showers' (not ingested as separate column)
   Pipeline co

## Step 7: Quarantine Outliers (Don't Drop Them)

In [117]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 7: Quarantine outliers — route suspicious rows to a separate Delta table
# ═══════════════════════════════════════════════════════════════════════════════

valid_wmo_list = list(VALID_WMO_CODES)

quarantine_condition = (
    (F.col("temperature_2m") < -20) | (F.col("temperature_2m") > 40) |
    (F.col("apparent_temperature") < -30) | (F.col("apparent_temperature") > 50) |
    (F.col("precipitation") > 100) |
    ((F.col("pressure_msl") >= 950) & (F.col("pressure_msl") < 970)) |
    ((F.col("pressure_msl") > 1040) & (F.col("pressure_msl") <= 1060)) |
    (~F.col("weather_code").isin(valid_wmo_list))
)

# Tag quarantined rows with reason(s) for human review
df = df.withColumn(
    "_quarantine_reasons",
    F.concat_ws("; ",
        F.when((F.col("temperature_2m") < -20) | (F.col("temperature_2m") > 40),
               F.concat(F.lit("temperature_2m="), F.col("temperature_2m").cast("string"))),
        F.when((F.col("apparent_temperature") < -30) | (F.col("apparent_temperature") > 50),
               F.concat(F.lit("apparent_temp="), F.col("apparent_temperature").cast("string"))),
        F.when(F.col("precipitation") > 100,
               F.concat(F.lit("precipitation="), F.col("precipitation").cast("string"))),
        F.when(((F.col("pressure_msl") >= 950) & (F.col("pressure_msl") < 970)) |
               ((F.col("pressure_msl") > 1040) & (F.col("pressure_msl") <= 1060)),
               F.concat(F.lit("pressure_msl="), F.col("pressure_msl").cast("string"))),
        F.when(~F.col("weather_code").isin(valid_wmo_list),
               F.concat(F.lit("unknown_wmo_code="), F.col("weather_code").cast("string"))),
    )
)

df_quarantine = df.filter(quarantine_condition).withColumn(
    "quarantined_at", F.current_timestamp()
)
df_clean = df.filter(~quarantine_condition)

clean_count = df_clean.count()
quarantine_count = df_quarantine.count()
total = clean_count + quarantine_count

print(f"Step 7: Quarantine split complete")
print(f"   Clean rows:       {clean_count:,} ({clean_count/total*100:.1f}%)")
print(f"   Quarantined rows: {quarantine_count:,} ({quarantine_count/total*100:.1f}%)")

if quarantine_count > 0:
    print(f"\n   Quarantine reasons breakdown:")
    df_quarantine.select("_quarantine_reasons").show(20, truncate=False)

df_clean = df_clean.drop("_quarantine_reasons")

StatementMeta(, 02149e88-cb61-466f-89b6-c94d46ee6b43, 119, Finished, Available, Finished, False)

Step 7: Quarantine split complete
   Clean rows:       13,320 (100.0%)
   Quarantined rows: 0 (0.0%)


## Step 8: Create `weather_category` Column

In [118]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 8: Create weather_category from WMO weather_code
# ═══════════════════════════════════════════════════════════════════════════════

df_clean = df_clean.withColumn(
    "weather_category",
    F.when(F.col("weather_code").isin([0, 1, 2, 3]), "Clear")
     .when(
         (F.col("weather_code") >= 4) & (F.col("weather_code") <= 49) &
         (~F.col("weather_code").isin([45, 48])),
         "Cloudy"
     )
     .when(F.col("weather_code").isin([45, 48]), "Fog")
     .when(F.col("weather_code").isin([51, 53, 55, 56, 57]), "Drizzle")
     .when(F.col("weather_code").isin([61, 63, 65, 66, 67, 80, 81, 82]), "Rain")
     .when(F.col("weather_code").isin([71, 73, 75, 77, 85, 86]), "Snow")
     .when(F.col("weather_code").isin([95, 96, 99]), "Thunderstorm")
     .otherwise("Unknown")
)

unknown_count = df_clean.filter(F.col("weather_category") == "Unknown").count()
if unknown_count > 0:
    print(f"WARNING: {unknown_count} rows mapped to 'Unknown' — check WMO code mapping")
    df_clean.filter(F.col("weather_category") == "Unknown")             .select("weather_code").distinct().show()

print("Step 8: weather_category created")
print("\n   Category distribution:")
df_clean.groupBy("weather_category")         .count()         .orderBy(F.desc("count"))         .show(truncate=False)

StatementMeta(, 02149e88-cb61-466f-89b6-c94d46ee6b43, 120, Finished, Available, Finished, False)

Step 8: weather_category created

   Category distribution:
+----------------+-----+
|weather_category|count|
+----------------+-----+
|Clear           |11681|
|Drizzle         |819  |
|Snow            |615  |
|Rain            |205  |
+----------------+-----+



## Step 9: Write to Silver Delta Table

In [119]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 9a: Drop Bronze-only columns before writing to Silver
# ═══════════════════════════════════════════════════════════════════════════════

BRONZE_ONLY_COLS = [
    "validation_reason", "ingest_run_id", "ingested_at",
    "ingest_date", "source",
]

existing_cols = set(df_clean.columns)
cols_to_drop = [c for c in BRONZE_ONLY_COLS if c in existing_cols]

if cols_to_drop:
    df_clean = df_clean.drop(*cols_to_drop)
    print(f"   Dropped Bronze-only columns: {cols_to_drop}")

print(f"\n   Silver schema ({len(df_clean.columns)} columns):")
df_clean.printSchema()

StatementMeta(, 02149e88-cb61-466f-89b6-c94d46ee6b43, 121, Finished, Available, Finished, False)

   Dropped Bronze-only columns: ['ingest_run_id', 'ingested_at', 'ingest_date', 'source']

   Silver schema (24 columns):
root
 |-- timestamp: timestamp (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- apparent_temperature: double (nullable = true)
 |-- relative_humidity_2m: double (nullable = true)
 |-- dew_point_2m: double (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- snowfall: double (nullable = true)
 |-- snow_depth: double (nullable = true)
 |-- wind_speed_10m: double (nullable = true)
 |-- wind_direction_10m: double (nullable = true)
 |-- wind_gusts_10m: double (nullable = true)
 |-- cloud_cover: double (nullable = true)
 |-- pressure_msl: double (nullable = true)
 |-- weather_code: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- timezone: string (nullable = true)
 |-- timezone_abbreviation: string (nullable = true)
 |-- elevation: double (

In [120]:
# ═══════════════════════════════════════════════════════════════════════════════
# STEP 9b: Write only NEW rows to silver_stg_weather (incremental append)
# ═══════════════════════════════════════════════════════════════════════════════

# ── Check if Silver table already exists ─────────────────────────────────────
try:
    spark.table(SILVER_TABLE).limit(1).collect()
    silver_exists = True
except Exception:
    silver_exists = False

if silver_exists:
    # Anti-join: keep only rows whose (borough, timestamp) are not already in Silver
    df_existing_keys = spark.table(SILVER_TABLE).select("borough", "timestamp")
    df_to_write = df_clean.join(
        df_existing_keys,
        on=["borough", "timestamp"],
        how="left_anti"
    )
    already_present = df_clean.count() - df_to_write.count()
    print(f"Silver table exists — incremental write mode")
    print(f"   Rows in this batch : {df_clean.count():,}")
    print(f"   Already in Silver  : {already_present:,} (skipped)")
    print(f"   New rows to write  : {df_to_write.count():,}")
else:
    df_to_write = df_clean
    print(f"Silver table does not exist — full initial load ({df_clean.count():,} rows)")

# ── Write new rows ────────────────────────────────────────────────────────────
# Count once before writing — df_to_write is lazy and re-evaluates on every .count() call
rows_to_write = df_to_write.count()

if rows_to_write > 0:
    df_to_write.write.format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(SILVER_TABLE)
    print(f"\n Written {rows_to_write:,} new rows → {SILVER_TABLE}")
else:
    print(f"\n No new rows to write — Silver table is already up to date")

silver_total = spark.table(SILVER_TABLE).count()
print(f"   Total rows in Silver now: {silver_total:,}")


# ── Write quarantine rows (same incremental logic) ────────────────────────────
if quarantine_count > 0:
    try:
        spark.table(QUARANTINE_TABLE).limit(1).collect()
        q_exists = True
    except Exception:
        q_exists = False

    if q_exists:
        q_existing_keys = spark.table(QUARANTINE_TABLE).select("borough", "timestamp")
        df_quarantine_new = df_quarantine.join(
            q_existing_keys, on=["borough", "timestamp"], how="left_anti"
        )
    else:
        df_quarantine_new = df_quarantine

    if df_quarantine_new.count() > 0:
        df_quarantine_new.write.format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable(QUARANTINE_TABLE)
        print(f"   Quarantine: {df_quarantine_new.count():,} new rows → {QUARANTINE_TABLE}")
    else:
        print(f"   Quarantine: no new rows")
else:
    print("\nNo quarantined rows in this batch")

StatementMeta(, 02149e88-cb61-466f-89b6-c94d46ee6b43, 122, Finished, Available, Finished, False)

Silver table exists — incremental write mode
   Rows in this batch : 13,320
   Already in Silver  : 13,270 (skipped)
   New rows to write  : 50

✅ Written 50 new rows → silver_stg_weather
   Total rows in Silver now: 13,320

No quarantined rows in this batch


## Summary
All 9 steps from the cleaning specification have been applied:

| Step | Status |
|------|--------|
| 1–2. Flatten + borough | Done in Bronze ingestion |
| 3. Verify timestamps + hourly gaps | Hard fail if gaps found |
| 4. NULL + range checks | Hard fail if violations |
| 5. Derive snowfall_mm (div 7) | Recalculated from Bronze's div 10 |
| 6. Cross-field validation | Hard fail if physics violated |
| 7. Quarantine outliers | Suspicious rows routed to quarantine table |
| 8. weather_category | WMO codes mapped (incl. Cloudy + snow showers) |
| 9. Write to Silver Delta | Written to Google Drive |

**Output tables:**
- `silver_stg_weather` — clean data, ready for Gold join
- `quarantine_weather` — suspicious rows for human review

In [121]:
# ═══════════════════════════════════════════════════════════════════════════════
# VERIFICATION QUERIES
# ── COLAB CHANGE: spark.read.format("delta").load() instead of spark.table() ──
# ═══════════════════════════════════════════════════════════════════════════════

silver = spark.table(SILVER_TABLE)
silver.createOrReplaceTempView("silver_stg_weather")

print("--- Row counts per borough ---")
spark.sql("SELECT borough, COUNT(*) as rows FROM silver_stg_weather GROUP BY borough ORDER BY borough").show()

print("--- Weather category distribution ---")
spark.sql("SELECT weather_category, COUNT(*) as rows FROM silver_stg_weather GROUP BY weather_category ORDER BY rows DESC").show()

print("--- Date range per borough ---")
spark.sql("""
    SELECT borough,
           MIN(timestamp) as earliest,
           MAX(timestamp) as latest,
           COUNT(DISTINCT DATE(timestamp)) as days
    FROM silver_stg_weather
    GROUP BY borough
    ORDER BY borough
""").show(truncate=False)

print("--- Sample rows ---")
silver.show(5, truncate=False)

StatementMeta(, 02149e88-cb61-466f-89b6-c94d46ee6b43, 123, Finished, Available, Finished, False)

--- Row counts per borough ---
+------------+----+
|     borough|rows|
+------------+----+
|       bronx|2664|
|    brooklyn|2664|
|   manhattan|2664|
|      queens|2664|
|statenisland|2664|
+------------+----+

--- Weather category distribution ---
+----------------+-----+
|weather_category| rows|
+----------------+-----+
|           Clear|11681|
|         Drizzle|  819|
|            Snow|  615|
|            Rain|  205|
+----------------+-----+

--- Date range per borough ---
+------------+-------------------+-------------------+----+
|borough     |earliest           |latest             |days|
+------------+-------------------+-------------------+----+
|bronx       |2026-01-01 00:00:00|2026-04-21 23:00:00|111 |
|brooklyn    |2026-01-01 00:00:00|2026-04-21 23:00:00|111 |
|manhattan   |2026-01-01 00:00:00|2026-04-21 23:00:00|111 |
|queens      |2026-01-01 00:00:00|2026-04-21 23:00:00|111 |
|statenisland|2026-01-01 00:00:00|2026-04-21 23:00:00|111 |
+------------+-------------------+----

## Validation

In [122]:
# Validation queries
bronze = spark.table(BRONZE_TABLE)
silver = spark.table(SILVER_TABLE)

total_bronze = bronze.count()
total_silver = silver.count()
print(f"Bronze: {total_bronze:,} rows")
print(f"Silver: {total_silver:,} rows")
print(f"Difference: {total_bronze - total_silver:,} (duplicates + quarantine)")

# Register for SQL queries
silver.createOrReplaceTempView("silver_stg_weather")

# Exactly 5 boroughs?
print("\n--- Borough row counts ---")
spark.sql("SELECT borough, COUNT(*) as rows FROM silver_stg_weather GROUP BY borough ORDER BY borough").show()

# Equal row counts per borough?
print("--- Date ranges ---")
spark.sql("""
    SELECT borough, MIN(timestamp) as earliest, MAX(timestamp) as latest,
           COUNT(*) as rows, COUNT(DISTINCT DATE(timestamp)) as days
    FROM silver_stg_weather GROUP BY borough ORDER BY borough
""").show(truncate=False)

# No duplicate (borough, timestamp) pairs?
print("--- Duplicate check ---")
spark.sql("""
    SELECT borough, timestamp, COUNT(*) as n
    FROM silver_stg_weather
    GROUP BY borough, timestamp
    HAVING COUNT(*) > 1
""").show()

# weather_category — no "Unknown"?
print("--- Category distribution ---")
spark.sql("SELECT weather_category, COUNT(*) as rows FROM silver_stg_weather GROUP BY weather_category ORDER BY rows DESC").show()

# snowfall_mm uses div 7?
print("--- Snowfall spot check ---")
spark.sql("SELECT snowfall, snowfall_mm, ROUND(snowfall/7, 4) as expected FROM silver_stg_weather WHERE snowfall > 0 LIMIT 3").show()

# Quarantine table
try:
    q = spark.read.format("delta").load(QUARANTINE_TABLE)
    print(f"\nQuarantine: {q.count()} rows")
    q.select("_quarantine_reasons").show(10, truncate=False)
except:
    print("\nNo quarantine table (0 outliers)")

StatementMeta(, 02149e88-cb61-466f-89b6-c94d46ee6b43, 124, Finished, Available, Finished, False)

Bronze: 13,320 rows
Silver: 13,320 rows
Difference: 0 (duplicates + quarantine)

--- Borough row counts ---
+------------+----+
|     borough|rows|
+------------+----+
|       bronx|2664|
|    brooklyn|2664|
|   manhattan|2664|
|      queens|2664|
|statenisland|2664|
+------------+----+

--- Date ranges ---
+------------+-------------------+-------------------+----+----+
|borough     |earliest           |latest             |rows|days|
+------------+-------------------+-------------------+----+----+
|bronx       |2026-01-01 00:00:00|2026-04-21 23:00:00|2664|111 |
|brooklyn    |2026-01-01 00:00:00|2026-04-21 23:00:00|2664|111 |
|manhattan   |2026-01-01 00:00:00|2026-04-21 23:00:00|2664|111 |
|queens      |2026-01-01 00:00:00|2026-04-21 23:00:00|2664|111 |
|statenisland|2026-01-01 00:00:00|2026-04-21 23:00:00|2664|111 |
+------------+-------------------+-------------------+----+----+

--- Duplicate check ---
+-------+---------+---+
|borough|timestamp|  n|
+-------+---------+---+
+-------+